# NLBS Phase 1 — GPU training on Colab (resumable)

**Before running:** `Runtime > Change runtime type > GPU`, then upload
`code.zip` and `data_bundle.zip` (built locally by `prepare_colab_bundle.sh`)
into a Google Drive folder named **`MyDrive/NLBS_Phase1/`**.

**Why this survives disconnects:** checkpoints/logs are written directly to
that Drive folder, not to Colab's temporary disk. If the runtime disconnects
or recycles, just re-run every cell — training auto-resumes from the last
completed epoch. You can also pull that Drive checkpoint back to your local
machine at any time and keep going there (see `NEXT_STEPS.md`).

## 1. Check GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv


## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/NLBS_Phase1'
assert os.path.isdir(DRIVE_ROOT), (
    f'{DRIVE_ROOT} not found. Create it and upload code.zip + data_bundle.zip '
    '(built by prepare_colab_bundle.sh) into it first.'
)
print('Drive contents:', os.listdir(DRIVE_ROOT))


## 3. Unzip code + data onto the fast local Colab disk

In [ ]:
import os

os.makedirs('/content/Phase1_DL', exist_ok=True)

!unzip -q -o "$DRIVE_ROOT/code.zip" -d /content/Phase1_DL
!unzip -q -o "$DRIVE_ROOT/data_bundle.zip" -d /content/nlbs_data_bundle

# prepare_colab_bundle.sh zips FROM inside its staging dir, so the archive's
# top-level entries are already 'nlbs_data/' and 'seed_checkpoint/' - reference
# them directly here rather than moving/renaming (shutil.move into an existing
# directory nests instead of replacing it, which silently breaks every path
# downstream, so don't do that).
LOCAL_DATA_DIR = '/content/nlbs_data_bundle/nlbs_data'
os.environ['NLBS_LOCAL_DATA_DIR'] = LOCAL_DATA_DIR
assert os.path.isdir(f'{LOCAL_DATA_DIR}/preproc_cache'), (
    f'{LOCAL_DATA_DIR}/preproc_cache missing - check data_bundle.zip was built '
    'correctly by prepare_colab_bundle.sh'
)
print('cache files:', len(os.listdir(f'{LOCAL_DATA_DIR}/preproc_cache')))
print('contents:', os.listdir(LOCAL_DATA_DIR))


## 4. Seed the Drive checkpoint (first run only)
Only copies the local checkpoint in if Drive has no training progress of its
own yet — safe to re-run this cell every session, it will never overwrite
further-along Colab progress.

In [ ]:
import os, shutil

seed_dir = '/content/nlbs_data_bundle/seed_checkpoint'
drive_ckpt_dir = os.path.join(DRIVE_ROOT, 'checkpoints')
os.makedirs(drive_ckpt_dir, exist_ok=True)

drive_has_progress = os.path.isfile(os.path.join(drive_ckpt_dir, 'last.pth'))
if drive_has_progress:
    print('Drive already has checkpoints/last.pth — leaving it as-is (not reseeding).')
elif os.path.isdir(seed_dir):
    for fn in os.listdir(seed_dir):
        shutil.copy(os.path.join(seed_dir, fn), os.path.join(drive_ckpt_dir, fn))
    print('Seeded Drive checkpoints/ from your local progress:', os.listdir(drive_ckpt_dir))
else:
    print('No seed checkpoint bundled — starting fresh.')


## 5. Install dependencies

In [ ]:
%cd /content/Phase1_DL
!pip install -q -r requirements.txt


## 6. Train
Safe to interrupt/disconnect at any point — checkpoints/metrics are on Drive.
Just re-run this cell (after remounting Drive and rerunning cells 3-4) to
resume from the last completed epoch.

In [ ]:
%cd /content/Phase1_DL
!NLBS_LOCAL_DATA_DIR="$NLBS_LOCAL_DATA_DIR" NLBS_DRIVE_ROOT="$DRIVE_ROOT" python run_colab_training.py


## 7. Live progress (run in a second cell while training, or after a restart)

In [ ]:
import pandas as pd
hist_path = f'{DRIVE_ROOT}/outputs/metrics_history.csv'
try:
    print(pd.read_csv(hist_path).tail(15).to_string(index=False))
except FileNotFoundError:
    print('No epochs completed yet.')


## 8. When you're satisfied: export final test results + Phase-2 features
Writes classification_report.pdf, confusion/ROC/PR/calibration plots, Grad-CAM
overlays, and the `.npy`/`.csv` feature files Phase 2 needs — all to
`DRIVE_ROOT/outputs/` (persistent, downloadable from Drive).

In [ ]:
import os
os.environ['NLBS_TEST_CHECKPOINT_DIR'] = f'{DRIVE_ROOT}/checkpoints'
os.environ['NLBS_TEST_OUTPUT_DIR'] = f'{DRIVE_ROOT}/outputs'
os.environ['NLBS_TEST_MANIFEST'] = f'{LOCAL_DATA_DIR}/patient_manifest.csv'
os.environ['NLBS_TEST_METADATA'] = f'{LOCAL_DATA_DIR}/metadata.csv'
os.environ['NLBS_TEST_CACHE_DIR'] = f'{LOCAL_DATA_DIR}/preproc_cache'
%cd /content/Phase1_DL
!python run_colab_test.py
